In [ ]:
# --- Vishalam setup: secrets -> clone private repo -> install ---
import os, subprocess, sys, time
T0 = time.time()
def hours_left(limit=11.2):  # Kaggle kills sessions at 12 h; keep a margin for the final upload
    return max(0.25, limit - (time.time() - T0) / 3600)
GITHUB_REPO = "shan3520/vishalam"
DATA_REPO = "shantanu3520/vishalam-data"
CKPT_REPO = "shantanu3520/vishalam"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except ImportError:
        from google.colab import userdata
        value = userdata.get(name)
    return (value or "").strip()  # pasted secrets often carry a stray space/newline

os.environ["HF_TOKEN"] = secret("HF_TOKEN")
GH_TOKEN = secret("GITHUB_TOKEN")

def github_check():
    """Explain a failed clone without ever printing the token."""
    import json, urllib.request, urllib.error
    req = urllib.request.Request(f"https://api.github.com/repos/{GITHUB_REPO}",
                                 headers={"Authorization": f"Bearer {GH_TOKEN}", "User-Agent": "vishalam"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            info = json.loads(resp.read())
            return f"API sees the repo (private={info.get('private')}); token works for the API"
    except urllib.error.HTTPError as e:
        return {401: "401: GitHub rejects this token (wrong/expired/revoked value in the secret)",
                403: "403: token valid but not allowed (check 'Contents: Read-only' permission)",
                404: f"404: token valid but cannot see {GITHUB_REPO} (give it access to this repo, "
                     "resource owner shan3520)"}.get(e.code, f"HTTP {e.code}")

def git(*args):
    import base64
    basic = base64.b64encode(f"x-access-token:{GH_TOKEN}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]  # token never in URLs or .git/config
    r = subprocess.run(["git", *auth, *args], capture_output=True, text=True)
    out = r.stdout + r.stderr
    for s in (GH_TOKEN, basic):
        if s:
            out = out.replace(s, "***")
    return r.returncode, out

os.chdir(WORK)
url = f"https://github.com/{GITHUB_REPO}.git"
if not os.path.isdir("vishalam"):
    code, out = git("clone", "--depth", "1", url, "vishalam")
    if code != 0:
        print(out)
        print("diagnosis:", github_check(), "| token length:", len(GH_TOKEN))
        raise SystemExit("git clone failed - see the diagnosis above")
else:
    git("-C", "vishalam", "pull", "--ff-only")
os.chdir(f"{WORK}/vishalam")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("ready:", subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
# --- fetch shared files from the private HF data repo ---
from huggingface_hub import HfApi, hf_hub_download
api = HfApi()
def fetch(name):
    return hf_hub_download(DATA_REPO, name, repo_type="dataset", local_dir=WORK)
def upload(local, name):
    api.upload_file(path_or_fileobj=local, path_in_repo=name, repo_id=DATA_REPO, repo_type="dataset")
TOKENIZER = fetch("tokenizer.json")


In [ ]:
# --- Build the M1 agent dataset (CPU; ~6-8 h for 100k episodes on 4 cores) ---
get_ipython().system(f"python -m vishalam.data.agent.build_m1 --out {WORK}/m1 --n-train 100000 --workers 4")
for name in ["train.jsonl", "test_seen.jsonl", "test_unseen.jsonl", "stats.json"]:
    upload(f"{WORK}/m1/{name}", f"m1/{name}")
print("uploaded m1/*")
